# ¿Existen los definidores natos?

"Ese delantero se come los goles." "Ese la mete siempre." Son afirmaciones sobre
una habilidad —**la definición**— que se da por supuesta pero casi nunca se mide.

Con un xG propio se puede medir. Si un jugador remató 100 veces desde sitios que
valían 12 goles y metió 17, le sacó 5 goles a sus ocasiones. La pregunta es si eso
**es una habilidad suya o le pasó**. Y hay una forma de distinguirlas: una
habilidad se repite; la suerte, no.

Este notebook cierra el arco de los tres anteriores. Usa el modelo de
`ml_xg_model.ipynb` y la maquinaria de `ml_style_persistence.ipynb`.

> **Cómo terminó esto**: la definición **existe** —la dispersión entre jugadores
> supera al azar con `z = +5.0`, no es un empate técnico— pero es **muy chica**.
> El 90% de lo que parece definición es ruido binomial; la habilidad real explica
> el 10%, con una desviación típica de **1.08 goles por cada 100 tiros**. Se
> repite de un año al otro con `r ≈ 0.10`, cuando una habilidad indiscutible como
> *generar tiros* se repite con `r ≈ 0.53`. Y hay que encoger la observación de un
> jugador al **13%** de su valor bruto antes de creérsela. Con eso hecho, los
> mejores son Mbappé y Kane — pero les sacan a sus ocasiones unos 2 goles por cada
> 100 tiros, no los 14 que sugieren las tablas sin corregir.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from xgboost import XGBClassifier

from viz_theme import (SEQUENTIAL_BLUE, DIVERGING, CLUSTER_COLORS, INK,
                       apply_plotly_theme, render_plot)

apply_plotly_theme(opaque_surface=True)
pd.set_option('display.max_columns', 40)
RNG = 42
rng = np.random.default_rng(0)

LARGO, ANCHO, PORTERIA = 105.0, 68.0, 7.32
SEASONS = ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

## 1. Un xG que no haya visto los goles del jugador

Acá hay una trampa que invalida el análisis entero si se pasa por alto.

Para juzgar si un jugador supera su xG, el xG **no puede venir de un modelo
entrenado con sus goles**. Si Mbappé está en el entrenamiento, el modelo ya
aprendió que los tiros desde donde él remata acaban en gol más a menudo, y le
sube el xG hasta hacerlo parecer promedio. La habilidad se autodestruye.

La solución es **validación cruzada por temporada**: para predecir 2023-24 se
entrena con las otras cuatro, y así con cada una. Cada tiro recibe un xG de un
modelo que nunca lo vio.

También se excluyen los penales (son una constante de 0.79 y los cobran los
especialistas, no es definición en juego) y los autogoles.

In [2]:
d = pd.read_csv('../data/processed/shots_all_seasons.csv', encoding='utf-8-sig')
d = d[(d.result != 'Own Goal') & (d.situation != 'Penalty')].copy()
d['gol'] = (d.result == 'Goal').astype(int)

px, py = d.location_x * LARGO, d.location_y * ANCHO
d['dist'] = np.hypot(LARGO - px, ANCHO / 2 - py)
a = np.hypot(LARGO - px, ANCHO / 2 + PORTERIA / 2 - py)
b = np.hypot(LARGO - px, ANCHO / 2 - PORTERIA / 2 - py)
d['angulo'] = np.arccos(np.clip((a ** 2 + b ** 2 - PORTERIA ** 2) / (2 * a * b), -1, 1))
d['dist_y'] = (d.location_y - 0.5).abs() * ANCHO
d['asistido'] = d.assist_player.notna().astype(int)
d['rebote'] = (d.groupby(['game_id', 'team', 'minute'])['shot_id'].transform('size') > 1).astype(int)

X = pd.get_dummies(d[['dist', 'angulo', 'dist_y', 'location_x', 'location_y',
                      'asistido', 'rebote', 'situation', 'body_part']],
                   columns=['situation', 'body_part'], dtype=float)
y = d.gol.values

d['xg_propio'] = np.nan
for s in SEASONS:
    fuera = (d.temporada == s).values
    modelo = XGBClassifier(n_estimators=800, learning_rate=0.05, max_depth=5,
                           min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                           reg_lambda=1.0, eval_metric='logloss', random_state=RNG, n_jobs=4)
    modelo.fit(X[~fuera], y[~fuera], verbose=False)
    d.loc[fuera, 'xg_propio'] = modelo.predict_proba(X[fuera])[:, 1]

print(f'{len(d)} tiros · {d.gol.sum()} goles\n')
print(f'xG propio (fuera de muestra): {d.xg_propio.sum():8.0f}  '
      f'sesgo {(d.xg_propio.sum()/d.gol.sum()-1)*100:+.1f}%')
print(f'xG de Understat             : {d.xg.sum():8.0f}  '
      f'sesgo {(d.xg.sum()/d.gol.sum()-1)*100:+.1f}%')

221922 tiros · 22099 goles

xG propio (fuera de muestra):    22172  sesgo +0.3%
xG de Understat             :    24750  sesgo +12.0%


El xG propio queda calibrado al **+0.3%** sobre 22.099 goles. El de Understat se
pasa un **12%** en estos mismos tiros, lo que confirma lo de
`ml_xg_model.ipynb` — y explica por qué no se puede usar el suyo acá: mediría
sobre todo su propio sesgo, y lo cargaría entero sobre los jugadores.

## 2. ¿Hay más dispersión de la que produce el azar?

Antes de buscar quiénes son los buenos definidores hay que comprobar que **haya
algo que buscar**.

Cada tiro es una moneda cargada: entra con probabilidad igual a su xG. Un jugador
con 50 tiros que suman 5 de xG puede meter 8 sin ser especial — es el equivalente
a sacar 8 caras en 50 tiradas de una moneda que sale cara el 10% de las veces.
Pasa.

La forma limpia de plantearlo: si la definición **no existiera**, la
sobre-actuación de cada jugador seguiría exactamente la dispersión binomial. Se
normaliza para que eso sea comparable:

$$z_i = \frac{G_i - \sum xG}{\sqrt{\sum xG(1-xG)}}$$

Si no hay habilidad, esos `z` tienen **desviación típica 1**. Si la hay,
más de 1: los buenos se van arriba, los malos abajo, y la nube se ensancha.

In [3]:
ps = d.groupby(['player', 'temporada']).agg(
    tiros=('gol', 'size'), goles=('gol', 'sum'),
    xg=('xg_propio', 'sum'),
    var=('xg_propio', lambda x: (x * (1 - x)).sum())).reset_index()
ps['sobre'] = ps.goles - ps.xg
ps['sobre100'] = ps.sobre / ps.tiros * 100
ps['z'] = ps.sobre / np.sqrt(ps['var'])

print(f'{len(ps)} jugador-temporada · {ps.player.nunique()} jugadores distintos')
print(f'tiros por jugador-temporada: mediana {ps.tiros.median():.0f}, '
      f'percentil 90 {ps.tiros.quantile(.9):.0f}\n')

p_all = d.xg_propio.values
grupos = d.groupby(['player', 'temporada']).indices

filas = []
for umbral in [20, 30, 50]:
    sub = ps[ps.tiros >= umbral]
    claves = list(zip(sub.player, sub.temporada))
    sd_sim = []
    for _ in range(200):                       # re-tirar cada tiro con su propia probabilidad
        sim = (rng.random(len(p_all)) < p_all).astype(int)
        zz = [(sim[grupos[k]].sum() - p_all[grupos[k]].sum()) /
              np.sqrt((p_all[grupos[k]] * (1 - p_all[grupos[k]])).sum()) for k in claves]
        sd_sim.append(np.std(zz))
    mu, sd = np.mean(sd_sim), np.std(sd_sim)
    filas.append({'Mínimo de tiros': umbral, 'Jugador-temporada': len(sub),
                  'SD observada': sub.z.std(), 'SD si fuera azar': mu,
                  'z del exceso': (sub.z.std() - mu) / sd})
pd.DataFrame(filas).round(3)

11165 jugador-temporada · 4551 jugadores distintos
tiros por jugador-temporada: mediana 13, percentil 90 47



,Mínimo de tiros,Jugador-temporada,SD observada,SD si fuera azar,z del exceso
0,20,4075,1.036,1.000,2.999
1,30,2516,1.076,1.000,4.978
2,50,1003,1.085,0.999,3.742


In [4]:
sub = ps[ps.tiros >= 30]
sim = (rng.random(len(p_all)) < p_all).astype(int)
claves = list(zip(sub.player, sub.temporada))
z_nulo = np.array([(sim[grupos[k]].sum() - p_all[grupos[k]].sum()) /
                   np.sqrt((p_all[grupos[k]] * (1 - p_all[grupos[k]])).sum()) for k in claves])

fig = go.Figure()
bins = dict(start=-4, end=4, size=0.25)
fig.add_trace(go.Histogram(x=z_nulo, xbins=bins, name='Si la definición no existiera',
                           marker=dict(color=INK['grid']), opacity=0.85,
                           hovertemplate='z %{x}<br>%{y} jugadores<extra></extra>'))
fig.add_trace(go.Histogram(x=sub.z, xbins=bins, name='Observado',
                           marker=dict(color=SEQUENTIAL_BLUE[5]), opacity=0.7,
                           hovertemplate='z %{x}<br>%{y} jugadores<extra></extra>'))
fig.update_layout(
    barmode='overlay',
    title=dict(text='La definición existe, pero apenas ensancha la nube',
               subtitle=dict(text='Sobre-actuación normalizada de cada jugador-temporada '
                                  '(≥30 tiros) · SD 1.076 contra 1.000')),
    xaxis=dict(title='z  (goles por encima de lo esperado, en desviaciones binomiales)'),
    yaxis=dict(title='Jugador-temporada'),
    legend=dict(orientation='h', y=-0.18, x=0))
render_plot(fig, width=800, height=480)

**La definición existe.** Con `z = +5.0` el exceso de dispersión no es un empate
técnico: la nube observada es genuinamente más ancha que la que produce el azar.

Pero conviene ver *cuánto* más ancha: 1.076 contra 1.000. Los dos histogramas se pisan
casi por completo. Traducido a varianza: si la dispersión observada es 1.076 y la del azar 1.000,
la habilidad aporta el **14%** y el azar el 86% restante. Existe, y es
minúscula. (La sección 4 llega a un 10% por otra vía — agregando por jugador
a lo largo de las cinco temporadas en vez de por jugador-temporada; que dos
estimaciones independientes caigan en el mismo orden de magnitud es buena
señal.)

## 3. La prueba decisiva: ¿se repite?

Que la nube sea ancha podría deberse a otras cosas (un jugador que remató un año
raro, diferencias que el xG no captura). La prueba de que es **habilidad del
jugador** es que se repita: quien sobre-define un año debería hacerlo el
siguiente.

Y para saber si el número que salga es alto o bajo hace falta una vara. Se usa la
mejor disponible: **el volumen de tiro**, medido sobre los mismos jugadores y las
mismas temporadas. Que un jugador genere muchos tiros es una habilidad que nadie
discute, así que su persistencia es el "así se ve una habilidad de verdad" contra
el que comparar.

In [5]:
filas = []
for umbral in [20, 30, 50]:
    A, B, C, D = [], [], [], []
    for s1, s2 in zip(SEASONS, SEASONS[1:]):
        m = (ps[(ps.temporada == s1) & (ps.tiros >= umbral)][['player', 'sobre100', 'tiros']]
             .merge(ps[(ps.temporada == s2) & (ps.tiros >= umbral)][['player', 'sobre100', 'tiros']],
                    on='player', suffixes=('_t', '_t1')))
        A += list(m.sobre100_t); B += list(m.sobre100_t1)
        C += list(m.tiros_t);    D += list(m.tiros_t1)
    filas.append({'Mínimo de tiros': umbral, 'Pares (t, t+1)': len(A),
                  'r — definición': np.corrcoef(A, B)[0, 1],
                  'r — volumen de tiro': np.corrcoef(C, D)[0, 1]})
tabla_pers = pd.DataFrame(filas)
tabla_pers.round(3)

,Mínimo de tiros,"Pares (t, t+1)",r — definición,r — volumen de tiro
0,20,1931,0.094,0.556
1,30,1128,0.137,0.533
2,50,378,0.096,0.463


In [6]:
A, B, C, D = [], [], [], []
for s1, s2 in zip(SEASONS, SEASONS[1:]):
    m = (ps[(ps.temporada == s1) & (ps.tiros >= 30)][['player', 'sobre100', 'tiros']]
         .merge(ps[(ps.temporada == s2) & (ps.tiros >= 30)][['player', 'sobre100', 'tiros']],
                on='player', suffixes=('_t', '_t1')))
    A += list(m.sobre100_t); B += list(m.sobre100_t1)
    C += list(m.tiros_t);    D += list(m.tiros_t1)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=(f'Definición — r = {np.corrcoef(A,B)[0,1]:.2f}',
                                    f'Volumen de tiro — r = {np.corrcoef(C,D)[0,1]:.2f}'))
for j, (xx, yy, color) in enumerate([(A, B, DIVERGING['neg']), (C, D, SEQUENTIAL_BLUE[5])], start=1):
    fig.add_trace(go.Scatter(x=xx, y=yy, mode='markers',
                             marker=dict(size=5.5, color=color, opacity=0.45,
                                         line=dict(width=0)),
                             hovertemplate='temporada t: %{x:.1f}<br>t+1: %{y:.1f}<extra></extra>',
                             showlegend=False), row=1, col=j)
    fit = np.poly1d(np.polyfit(xx, yy, 1))
    xs = np.array([min(xx), max(xx)])
    fig.add_trace(go.Scatter(x=xs, y=fit(xs), mode='lines',
                             line=dict(color=INK['secondary'], width=2, dash='dash'),
                             hoverinfo='skip', showlegend=False), row=1, col=j)
fig.update_xaxes(title_text='Temporada t', row=1, col=1)
fig.update_xaxes(title_text='Temporada t', row=1, col=2)
fig.update_yaxes(title_text='Temporada t+1', row=1, col=1)
fig.update_yaxes(title_text='Temporada t+1', row=1, col=2)
fig.update_layout(
    title=dict(text='Así se ve una habilidad, y así se ve la definición',
               subtitle=dict(text='Mismos jugadores, mismas temporadas · izquierda goles sobre '
                                  'lo esperado por 100 tiros, derecha tiros por temporada')),
    margin=dict(t=150))
render_plot(fig, width=900, height=470)

**`r = 0.14` contra `r = 0.53`.** La nube de la izquierda no tiene forma; la de la
derecha es una diagonal clara.

Un jugador que sobre-definió el año pasado te dice **casi nada** sobre lo que va a
hacer este. El mismo jugador, si tiró mucho, va a volver a tirar mucho. Esa es la
diferencia entre una habilidad y un resultado.

## 4. ¿Cuánto hay que creerse de lo que se observa?

Sabiendo que la mayor parte de la sobre-actuación es ruido, la pregunta práctica
es: dado que un jugador lleva +6 goles sobre su xG, **¿cuánto de eso es él?**

La respuesta es el **encogimiento** (*empirical Bayes*): se estima cuánta varianza
hay de verdad entre jugadores y cuánta produce el azar, y se acerca cada
observación a la media en proporción a lo ruidosa que sea. Un jugador con 30
tiros se encoge casi del todo; uno con 600, mucho menos.

In [7]:
pj = d.groupby('player').agg(
    tiros=('gol', 'size'), goles=('gol', 'sum'), xg=('xg_propio', 'sum'),
    var=('xg_propio', lambda x: (x * (1 - x)).sum()),
    temporadas=('temporada', 'nunique')).reset_index()
pj = pj[pj.tiros >= 30].copy()

pj['bruto100'] = (pj.goles - pj.xg) / pj.tiros * 100      # lo que se ve
pj['var_muestral'] = pj['var'] / pj.tiros ** 2            # cuánto de eso es azar

var_total = ((pj.goles - pj.xg) / pj.tiros).var()
var_ruido = pj.var_muestral.mean()
tau2 = max(var_total - var_ruido, 0)                      # varianza de la habilidad real

print(f'{len(pj)} jugadores con >=30 tiros en las 5 temporadas\n')
print(f'  varianza total observada : {var_total:.6f}')
print(f'  varianza que produce azar: {var_ruido:.6f}   ({var_ruido/var_total:.0%})')
print(f'  varianza de HABILIDAD    : {tau2:.6f}   ({tau2/var_total:.0%})')
print(f'\n  desviación típica de la habilidad: {np.sqrt(tau2)*100:.2f} goles por 100 tiros')

pj['credibilidad'] = tau2 / (tau2 + pj.var_muestral)
pj['skill100'] = pj.bruto100 * pj.credibilidad

print(f'\ncredibilidad media: {pj.credibilidad.mean():.2f}  (1 = creerse todo, 0 = todo ruido)')
for lo, hi, et in [(30, 50, '30-50 tiros'), (100, 200, '100-200'), (400, 9999, 'más de 400')]:
    s = pj[(pj.tiros >= lo) & (pj.tiros < hi)]
    print(f'   {et:14s}: {s.credibilidad.mean():.2f}')

2002 jugadores con >=30 tiros en las 5 temporadas

  varianza total observada : 0.001198
  varianza que produce azar: 0.001082   (90%)
  varianza de HABILIDAD    : 0.000117   (10%)

  desviación típica de la habilidad: 1.08 goles por 100 tiros

credibilidad media: 0.13  (1 = creerse todo, 0 = todo ruido)
   30-50 tiros   : 0.07
   100-200       : 0.19
   más de 400    : 0.38


In [8]:
orden = pj.sort_values('tiros')
fig = go.Figure(go.Scatter(
    x=orden.tiros, y=orden.credibilidad, mode='markers',
    marker=dict(size=5, color=SEQUENTIAL_BLUE[5], opacity=0.5, line=dict(width=0)),
    customdata=orden.player,
    hovertemplate='<b>%{customdata}</b><br>%{x} tiros<br>credibilidad %{y:.2f}<extra></extra>'))
for yv, txt in [(0.5, 'la mitad es señal')]:
    fig.add_hline(y=yv, line=dict(color=INK['axis'], width=1, dash='dash'))
    fig.add_annotation(x=pj.tiros.max(), y=yv, text=txt, showarrow=False,
                       xanchor='right', yanchor='bottom',
                       font=dict(size=10.5, color=INK['muted']))
fig.update_layout(
    title=dict(text='Cuántos tiros hacen falta para creerle a un delantero',
               subtitle=dict(text='Fracción de la sobre-actuación observada que es habilidad '
                                  'y no azar, según el tamaño de muestra')),
    xaxis=dict(title='Tiros en cinco temporadas'),
    yaxis=dict(title='Credibilidad', range=[0, 0.6]),
    showlegend=False)
render_plot(fig, width=780, height=470)

**El 90% de la variación entre jugadores es azar y el 10% es habilidad.** La
habilidad real tiene una desviación típica de **1.08 goles por cada 100 tiros**:
un delantero excepcional, a un desvío por encima, le saca **un gol por temporada**
a sus ocasiones respecto de uno promedio.

Y la credibilidad media es **0.13**. O sea: de los +6 goles que veas en una tabla,
créete menos de uno. Ni siquiera los jugadores con más de 400 tiros en cinco años
llegan a que la mitad de lo observado sea señal.

## 5. Entonces, ¿quiénes son los mejores?

In [9]:
cols = ['player', 'tiros', 'goles', 'xg', 'bruto100', 'credibilidad', 'skill100']
top = pj.nlargest(12, 'skill100')[cols].rename(columns={
    'player': 'Jugador', 'tiros': 'Tiros', 'goles': 'Goles', 'xg': 'xG',
    'bruto100': 'Bruto /100', 'credibilidad': 'Credib.', 'skill100': 'Habilidad /100'})
top.round(2).reset_index(drop=True)

,Jugador,Tiros,Goles,xG,Bruto /100,Credib.,Habilidad /100
0,Kylian Mbappe-Lottin,690,113,77.95,5.08,0.48,2.43
1,Harry Kane,610,112,83.89,4.61,0.42,1.94
2,Serge Gnabry,264,45,27.03,6.81,0.28,1.89
3,Julian Brandt,191,37,21.54,8.09,0.22,1.74
4,Maximilian Arnold,141,12,4.90,5.04,0.34,1.71
5,Franck Honorat,182,26,14.03,6.58,0.25,1.66
6,Phil Foden,317,53,36.08,5.34,0.30,1.61
7,Jonas Hofmann,190,33,19.07,7.33,0.21,1.57
8,Kevin De Bruyne,268,32,19.76,4.57,0.34,1.53
9,Son Heung-Min,305,54,37.42,5.44,0.27,1.48


In [10]:
peores = pj.nsmallest(8, 'skill100')[cols].rename(columns={
    'player': 'Jugador', 'tiros': 'Tiros', 'goles': 'Goles', 'xg': 'xG',
    'bruto100': 'Bruto /100', 'credibilidad': 'Credib.', 'skill100': 'Habilidad /100'})
peores.round(2).reset_index(drop=True)

,Jugador,Tiros,Goles,xG,Bruto /100,Credib.,Habilidad /100
0,Dominic Calvert-Lewin,249,22,38.49,-6.62,0.22,-1.48
1,Florian Sotoca,279,15,25.92,-3.91,0.31,-1.20
2,Gerard Gumbau,146,1,6.92,-4.06,0.29,-1.18
3,Christian Eriksen,121,3,10.31,-6.04,0.18,-1.08
4,Carles Aleñá,144,4,10.85,-4.76,0.22,-1.03
5,Joe Willock,135,6,14.50,-6.30,0.16,-1.02
6,Pablo Ibáñez,114,3,9.41,-5.62,0.18,-1.01
7,Angelo Fulgini,150,8,15.72,-5.15,0.18,-0.94


Los nombres son la mejor validación que puede tener el método: **Mbappé y Kane**
arriba, y abajo un delantero con fama pública de fallador. No se le dijo al modelo
quién es bueno; salió de encoger 22.099 goles contra su xG.

Ahora bien, conviene mirar la magnitud: Mbappé, el mejor definidor de las cinco ligas en cinco
temporadas, le saca **2.4 goles por cada 100 tiros** a sus ocasiones. Con 690
tiros, son unos 17 goles en cinco años — real y valioso, pero muy lejos de la
mitología del "killer del área".

### Y lo que pasaría sin encoger

In [11]:
bruto = pj.nlargest(10, 'bruto100')[cols].rename(columns={
    'player': 'Jugador', 'tiros': 'Tiros', 'goles': 'Goles', 'xg': 'xG',
    'bruto100': 'Bruto /100', 'credibilidad': 'Credib.', 'skill100': 'Habilidad /100'})
bruto.round(2).reset_index(drop=True)

,Jugador,Tiros,Goles,xG,Bruto /100,Credib.,Habilidad /100
0,Tajon Buchanan,38,9,3.47,14.55,0.06,0.91
1,Manor Solomon,31,7,2.55,14.37,0.05,0.71
2,Masaya Okugawa,32,8,3.77,13.23,0.04,0.54
3,Dries Mertens,44,10,4.31,12.94,0.06,0.80
4,Bafode Diakite,52,12,5.99,11.56,0.06,0.75
5,Enrico Del Prato,30,6,2.72,10.93,0.04,0.49
6,Alberth Elis,34,9,5.33,10.80,0.04,0.45
7,Elijah Adebayo,31,10,6.67,10.76,0.03,0.28
8,Fabien Centonze,37,7,3.02,10.76,0.07,0.73
9,Rasmus Kristensen,61,11,4.55,10.57,0.10,1.07


**Esta tabla es la razón de ser de todo el notebook.** Ordenando por la
sobre-actuación bruta —que es lo que hace cualquier ranking de "mejores
definidores" que circula por ahí— el podio se llena de jugadores con 30 o 40
tiros que tuvieron un buen rato.

Tajon Buchanan aparece primero con **+14.6 goles por 100 tiros**: metió 9 con 3.47
de xG. En 38 tiros. Eso es una racha, no una habilidad, y el encogimiento lo baja
a 0.91 — por debajo de una docena de jugadores con muestras de verdad.

Es exactamente el modo de fallo que tiene cualquier método que busque *rarezas*
sin mirar el **tamaño de muestra**: detectar como élite a quien tuvo suerte con
pocos intentos. La corrección no es un detalle técnico, es lo que separa un
ranking de un sinsentido.

## Conclusión

1. **La definición existe.** La dispersión entre jugadores supera la binomial con
   `z = +5.0`, medido contra un nulo por simulación que re-tira los 221.922 tiros
   con su propia probabilidad. No es un empate técnico.
2. **Pero es diminuta.** Explica el 10% de la variación entre jugadores; el otro
   90% es azar. Su desviación típica es de **1.08 goles por 100 tiros**.
3. **Y apenas se repite**: `r ≈ 0.10-0.14` de una temporada a la siguiente, contra
   `r ≈ 0.53` del volumen de tiro sobre los mismos jugadores. Saber que alguien
   sobre-definió el año pasado casi no ayuda a predecir el que viene.
4. **Hay que encoger al 13%** lo que se observa. Sin eso, cualquier ranking de
   definidores es una lista de rachas con muestra chica.
5. **Con la corrección hecha, los nombres son los correctos** — Mbappé, Kane,
   Gnabry — y las magnitudes, mucho más modestas de lo que dice el folklore: el
   mejor de cinco ligas y cinco temporadas gana ~2.4 goles por cada 100 tiros.

**La lección transferible**: *generar* ocasiones es una habilidad estable y
medible; *convertirlas* por encima de lo esperado es casi todo ruido. Para juzgar
a un delantero, el xG que genera dice mucho más que la diferencia entre sus goles
y su xG.

**Limitaciones**: el xG propio no ve al portero ni a los defensores, así que parte
de lo que acá se llama "definición" podría ser en realidad seleccionar mejores
momentos dentro de una misma casilla de xG. Y no se modela el cambio de un
jugador a lo largo de los años: se le asigna una habilidad fija en cinco
temporadas, lo que subestima a quien mejoró o empeoró de verdad.